In [5]:
import os
import random

import cv2
import numpy as np
from scipy import ndimage
from tqdm import tqdm


# ==========================
# 設定
# ==========================

green_image_dir = "preprocessed_green_images"
original_image_dir = "healthy_images"

output_dir = "vessel_output"
mask_output_dir = os.path.join(output_dir, "masks")
skeleton_output_dir = os.path.join(output_dir, "skeletons")
overlay_output_dir = os.path.join(output_dir, "overlays")

# ランダム確認用出力フォルダ
random_sample_output_dir = os.path.join(output_dir, "random_samples")


# ==========================
# ランダム確認用設定
# ==========================

RANDOM_SAMPLE_COUNT = 10
RANDOM_SAMPLE_SEED = 42


# ==========================
# チューニング用パラメータ
# ==========================

MAX_PROCESSING_DIMENSION = 800

BLACKHAT_KERNEL_SIZES = [7, 11, 15, 21]

FRANGI_SIGMAS = [0.75, 1.25, 1.75, 2.5, 4]

HYSTERESIS_LOW_PERCENTILE = 78
HYSTERESIS_HIGH_PERCENTILE = 93
HYSTERESIS_SEED_PERCENTILE = 62
HYSTERESIS_MIN_COMPONENT_AREA = 12

PRUNE_MIN_BRANCH_LENGTH = 12
PRUNE_MIN_MEAN_SCORE = 26

ENABLE_BRIDGE_GAPS = False
BRIDGE_MAX_GAP = 8
BRIDGE_MAX_ANGLE_DEG = 20
BRIDGE_LOOKBACK = 6

FOV_ERODE_PX = 12

MIN_COMPONENT_AREA_AFTER_CONNECT = 12
MIN_SKELETON_COMPONENT_PIXELS = 6

ELONGATION_MIN_AREA_TO_CHECK = 60
ELONGATION_MIN_ASPECT_RATIO = 1.8

EXTRA_CLAHE_CLIP_LIMIT = 1.6
EXTRA_CLAHE_TILE_GRID_SIZE = (8, 8)


# ==========================
# ファイル名変換
# ==========================

def get_original_file_name(preprocessed_file):
    if preprocessed_file.endswith("_green_preprocessed.png"):
        image_id = preprocessed_file.replace("_green_preprocessed.png", "")
    else:
        image_id = os.path.splitext(preprocessed_file)[0]

    return image_id + ".png"


def get_image_id(preprocessed_file):
    if preprocessed_file.endswith("_green_preprocessed.png"):
        return preprocessed_file.replace("_green_preprocessed.png", "")

    return os.path.splitext(preprocessed_file)[0]


# ==========================
# 縮小・拡大
# ==========================

def resize_for_processing(image, max_dimension=MAX_PROCESSING_DIMENSION):
    """
    長辺が max_dimension を超える場合のみ縮小する。
    Black-hat / Frangi は重いため、ここで高速化する。
    """

    h, w = image.shape[:2]
    longest_side = max(h, w)

    if longest_side <= max_dimension:
        return image, 1.0

    scale = max_dimension / longest_side
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))

    resized = cv2.resize(
        image,
        (new_w, new_h),
        interpolation=cv2.INTER_AREA
    )

    return resized, scale


def resize_to_shape(binary_or_gray_image, target_shape, is_binary=True):
    """
    処理解像度の画像を元の解像度に戻す。
    2値画像は INTER_NEAREST を使い、境界のぼけを防ぐ。
    """

    target_h, target_w = target_shape[:2]

    interpolation = cv2.INTER_NEAREST if is_binary else cv2.INTER_LINEAR

    resized = cv2.resize(
        binary_or_gray_image,
        (target_w, target_h),
        interpolation=interpolation
    )

    if is_binary:
        _, resized = cv2.threshold(
            resized,
            127,
            255,
            cv2.THRESH_BINARY
        )

    return resized


# ==========================
# Percentile 正規化
# ==========================

def percentile_normalize(image, lower_percentile=1, upper_percentile=99):
    """
    外れ値の影響を抑えて 0-255 に正規化する。
    """

    output = np.zeros_like(image, dtype=np.uint8)
    values = image[image > 0]

    if values.size == 0:
        return output

    low = np.percentile(values, lower_percentile)
    high = np.percentile(values, upper_percentile)

    if high <= low:
        return output

    clipped = np.clip(image, low, high)
    normalized = (clipped - low) / (high - low) * 255
    normalized = normalized.astype(np.uint8)

    output[image > 0] = normalized[image > 0]

    return output


# ==========================
# FOVマスク
# ==========================

def compute_fov_mask(green, erode_px=FOV_ERODE_PX):
    """
    有効な眼底領域のマスクを作成する。
    境界付近は誤検出が出やすいため、少し内側に収縮させる。
    """

    _, raw_mask = cv2.threshold(
        green,
        10,
        255,
        cv2.THRESH_BINARY
    )

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        raw_mask,
        connectivity=8
    )

    if num_labels <= 1:
        return raw_mask

    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])

    fov_mask = np.zeros_like(raw_mask)
    fov_mask[labels == largest_label] = 255

    if erode_px > 0:
        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (erode_px * 2 + 1, erode_px * 2 + 1)
        )
        fov_mask = cv2.erode(fov_mask, kernel)

    return fov_mask


# ==========================
# コントラスト強調
# ==========================

def enhance_contrast(green):
    """
    血管抽出用の追加CLAHE。
    前処理側でもCLAHE済みなので、ここでは中程度にする。
    """

    clahe = cv2.createCLAHE(
        clipLimit=EXTRA_CLAHE_CLIP_LIMIT,
        tileGridSize=EXTRA_CLAHE_TILE_GRID_SIZE
    )

    return clahe.apply(green)


# ==========================
# Frangi Filter
# ==========================

def frangi_vesselness(green, sigmas=FRANGI_SIGMAS):
    """
    線状構造を強調する Frangi filter。
    black_ridges=True により暗い線状構造を血管候補として強調する。
    """

    try:
        from skimage.filters import frangi
    except Exception:
        print("skimage が使えないため Frangi filter をスキップします。")
        return np.zeros_like(green, dtype=np.uint8)

    image_float = green.astype(np.float32) / 255.0

    vesselness = frangi(
        image_float,
        sigmas=sigmas,
        black_ridges=True
    )

    vesselness = np.nan_to_num(vesselness)

    if vesselness.max() <= vesselness.min():
        return np.zeros_like(green, dtype=np.uint8)

    vesselness = (
        (vesselness - vesselness.min())
        / (vesselness.max() - vesselness.min())
        * 255
    )

    return vesselness.astype(np.uint8)


# ==========================
# Hysteresis Threshold
# ==========================

def hysteresis_threshold_with_seed(
    score_image,
    seed_image,
    low_percentile=HYSTERESIS_LOW_PERCENTILE,
    high_percentile=HYSTERESIS_HIGH_PERCENTILE,
    seed_percentile=HYSTERESIS_SEED_PERCENTILE,
    min_component_area=HYSTERESIS_MIN_COMPONENT_AREA,
):
    """
    弱い候補の中で、強い候補につながっている成分だけを残す。
    """

    score_values = score_image[score_image > 0]
    seed_values = seed_image[seed_image > 0]

    if score_values.size == 0:
        return np.zeros_like(score_image, dtype=np.uint8)

    if seed_values.size == 0:
        seed_values = score_values

    low_th = np.percentile(score_values, low_percentile)
    high_th = np.percentile(score_values, high_percentile)
    seed_th = np.percentile(seed_values, seed_percentile)

    low_mask = np.zeros_like(score_image, dtype=np.uint8)
    strong_mask = np.zeros_like(score_image, dtype=np.uint8)

    low_mask[score_image >= low_th] = 255
    strong_mask[
        (score_image >= high_th)
        & (seed_image >= seed_th)
    ] = 255

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        low_mask,
        connectivity=8
    )

    output = np.zeros_like(score_image, dtype=np.uint8)

    for label in range(1, num_labels):
        area = stats[label, cv2.CC_STAT_AREA]

        if area < min_component_area:
            continue

        component = labels == label

        if np.any(strong_mask[component] > 0):
            output[component] = 255

    return output


# ==========================
# 線状カーネルで接続
# ==========================

def connect_vessel_segments(binary_image):
    """
    途切れた血管候補を線状カーネルで軽く接続する。
    網目状の誤接続を避けるため、強すぎない設定にしている。
    """

    kernels = []

    size = 5
    center = size // 2

    for angle_deg in [0, 45, 90, 135]:
        kernel = np.zeros((size, size), np.uint8)

        theta = np.deg2rad(angle_deg)
        dx = np.cos(theta)
        dy = np.sin(theta)

        for t in range(-center, center + 1):
            x = int(round(center + t * dx))
            y = int(round(center + t * dy))

            if 0 <= x < size and 0 <= y < size:
                kernel[y, x] = 1

        kernels.append(kernel)

    connected = binary_image.copy()

    for kernel in kernels:
        closed = cv2.morphologyEx(
            binary_image,
            cv2.MORPH_CLOSE,
            kernel
        )
        connected = cv2.bitwise_or(connected, closed)

    return connected


# ==========================
# 小領域除去
# ==========================

def remove_tiny_components(binary_image, min_area=MIN_COMPONENT_AREA_AFTER_CONNECT):
    """
    小さい孤立成分を除去する。
    """

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary_image,
        connectivity=8
    )

    output = np.zeros_like(binary_image)

    for label in range(1, num_labels):
        area = stats[label, cv2.CC_STAT_AREA]

        if area >= min_area:
            output[labels == label] = 255

    return output


# ==========================
# 細長さフィルタ
# ==========================

def filter_by_elongation(
    binary_image,
    min_area_to_check=ELONGATION_MIN_AREA_TO_CHECK,
    min_aspect_ratio=ELONGATION_MIN_ASPECT_RATIO,
):
    """
    面積が小さく、かつ丸に近い成分を除去する。
    血管は基本的に細長いため、小さい丸い成分はノイズとみなす。
    """

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary_image,
        connectivity=8
    )

    output = np.zeros_like(binary_image)

    for label in range(1, num_labels):
        area = stats[label, cv2.CC_STAT_AREA]

        if area >= min_area_to_check:
            output[labels == label] = 255
            continue

        component_mask = (labels == label).astype(np.uint8) * 255

        contours, _ = cv2.findContours(
            component_mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        if len(contours) == 0:
            continue

        contour = max(contours, key=cv2.contourArea)

        if len(contour) < 5:
            output[labels == label] = 255
            continue

        (_, _), (w, h), _ = cv2.minAreaRect(contour)

        short_side = min(w, h)
        long_side = max(w, h)

        aspect_ratio = long_side / short_side if short_side > 0 else 0

        if aspect_ratio >= min_aspect_ratio:
            output[labels == label] = 255

    return output


# ==========================
# 細線化
# ==========================

def skeletonize_binary(binary_image):
    """
    2値血管マスクを 1px 幅のスケルトンに変換する。
    """

    binary = binary_image > 0

    try:
        from skimage.morphology import skeletonize

        skeleton = skeletonize(binary)
        return skeleton.astype(np.uint8) * 255

    except Exception:
        pass

    img = binary.astype(np.uint8) * 255
    skeleton = np.zeros_like(img)

    element = cv2.getStructuringElement(
        cv2.MORPH_CROSS,
        (3, 3)
    )

    while True:
        opened = cv2.morphologyEx(
            img,
            cv2.MORPH_OPEN,
            element
        )

        temp = cv2.subtract(img, opened)
        eroded = cv2.erode(img, element)

        skeleton = cv2.bitwise_or(skeleton, temp)
        img = eroded.copy()

        if cv2.countNonZero(img) == 0:
            break

    return skeleton


# ==========================
# スケルトン共通処理
# ==========================

def get_skeleton_neighbors(y, x, skeleton_bool):
    """
    スケルトン画素の8近傍を取得する。
    """

    h, w = skeleton_bool.shape
    neighbors = []

    for dy in [-1, 0, 1]:
        for dx in [-1, 0, 1]:
            if dy == 0 and dx == 0:
                continue

            ny = y + dy
            nx = x + dx

            if 0 <= ny < h and 0 <= nx < w and skeleton_bool[ny, nx]:
                neighbors.append((ny, nx))

    return neighbors


def prune_short_skeleton_branches(
    skeleton_image,
    vessel_score,
    min_branch_length=PRUNE_MIN_BRANCH_LENGTH,
    min_mean_score=PRUNE_MIN_MEAN_SCORE,
):
    """
    短く、かつ血管スコアが低い枝を除去する。
    """

    skeleton_bool = skeleton_image > 0

    if np.sum(skeleton_bool) == 0:
        return skeleton_image.copy()

    neighbor_kernel = np.array(
        [
            [1, 1, 1],
            [1, 0, 1],
            [1, 1, 1]
        ],
        dtype=np.uint8
    )

    neighbor_count = ndimage.convolve(
        skeleton_bool.astype(np.uint8),
        neighbor_kernel,
        mode="constant"
    )

    degree = (neighbor_count * skeleton_bool).astype(np.uint8)

    endpoints = list(zip(*np.where(skeleton_bool & (degree == 1))))

    remove_mask = np.zeros_like(skeleton_image, dtype=bool)
    visited_edges = set()

    for endpoint in endpoints:
        path = []
        current = endpoint
        previous = None

        while True:
            cy, cx = current
            path.append(current)

            neighbors = get_skeleton_neighbors(
                cy,
                cx,
                skeleton_bool
            )

            if previous is not None:
                neighbors = [
                    nb for nb in neighbors
                    if nb != previous
                ]

            if len(neighbors) == 0:
                break

            if previous is not None and degree[cy, cx] != 2:
                break

            next_pixel = neighbors[0]
            edge_key = tuple(sorted([current, next_pixel]))

            if edge_key in visited_edges:
                break

            visited_edges.add(edge_key)

            previous = current
            current = next_pixel

            if len(path) > 10000:
                break

        if len(path) == 0:
            continue

        path_y = np.array([p[0] for p in path])
        path_x = np.array([p[1] for p in path])

        branch_length = len(path)
        mean_score = float(np.mean(vessel_score[path_y, path_x]))

        if branch_length < min_branch_length and mean_score < min_mean_score:
            remove_mask[path_y, path_x] = True

    pruned = skeleton_image.copy()
    pruned[remove_mask] = 0

    return pruned


def remove_tiny_skeleton_components(
    skeleton_image,
    min_pixels=MIN_SKELETON_COMPONENT_PIXELS
):
    """
    スケルトン化後の小さい孤立成分を除去する。
    """

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        skeleton_image,
        connectivity=8
    )

    output = np.zeros_like(skeleton_image)

    for label in range(1, num_labels):
        area = stats[label, cv2.CC_STAT_AREA]

        if area >= min_pixels:
            output[labels == label] = 255

    return output


# ==========================
# ギャップ橋渡し
# ==========================

def bridge_skeleton_gaps(
    skeleton_image,
    max_gap=BRIDGE_MAX_GAP,
    max_angle_deg=BRIDGE_MAX_ANGLE_DEG,
    lookback=BRIDGE_LOOKBACK,
):
    """
    近接していて向きが一致する端点同士を接続する。
    現在は ENABLE_BRIDGE_GAPS=False のため通常は使わない。
    """

    skeleton_bool = skeleton_image > 0
    ys, xs = np.where(skeleton_bool)

    if len(ys) == 0:
        return skeleton_image.copy()

    neighbor_kernel = np.array(
        [
            [1, 1, 1],
            [1, 0, 1],
            [1, 1, 1]
        ],
        dtype=np.uint8
    )

    neighbor_count = ndimage.convolve(
        skeleton_bool.astype(np.uint8),
        neighbor_kernel,
        mode="constant"
    )

    degree = (neighbor_count * skeleton_bool).astype(np.uint8)

    endpoints = list(zip(*np.where(skeleton_bool & (degree == 1))))

    def estimate_direction(endpoint):
        current = endpoint
        previous = None
        trail = [current]

        for _ in range(lookback):
            cy, cx = current

            neighbors = get_skeleton_neighbors(
                cy,
                cx,
                skeleton_bool
            )

            if previous is not None:
                neighbors = [
                    nb for nb in neighbors
                    if nb != previous
                ]

            if len(neighbors) == 0:
                break

            previous = current
            current = neighbors[0]
            trail.append(current)

        if len(trail) < 2:
            return None

        start = np.array(trail[-1], dtype=np.float32)
        end = np.array(trail[0], dtype=np.float32)

        vec = end - start
        norm = np.linalg.norm(vec)

        if norm == 0:
            return None

        return vec / norm

    bridged = skeleton_image.copy()
    used_endpoints = set()

    n = len(endpoints)

    for i in range(n):
        if i in used_endpoints:
            continue

        p1 = np.array(endpoints[i], dtype=np.float32)
        dir1 = estimate_direction(endpoints[i])

        if dir1 is None:
            continue

        best_j = None
        best_dist = max_gap + 1

        for j in range(n):
            if i == j or j in used_endpoints:
                continue

            p2 = np.array(endpoints[j], dtype=np.float32)
            dir2 = estimate_direction(endpoints[j])

            if dir2 is None:
                continue

            dist = np.linalg.norm(p2 - p1)

            if dist <= 0 or dist > max_gap:
                continue

            connect_vec = (p2 - p1) / dist

            angle1 = np.degrees(
                np.arccos(
                    np.clip(
                        np.dot(dir1, connect_vec),
                        -1.0,
                        1.0
                    )
                )
            )

            angle2 = np.degrees(
                np.arccos(
                    np.clip(
                        np.dot(dir2, -connect_vec),
                        -1.0,
                        1.0
                    )
                )
            )

            if angle1 > max_angle_deg:
                continue

            if angle2 > max_angle_deg:
                continue

            if dist < best_dist:
                best_dist = dist
                best_j = j

        if best_j is not None:
            p2 = endpoints[best_j]

            cv2.line(
                bridged,
                (int(p1[1]), int(p1[0])),
                (int(p2[1]), int(p2[0])),
                255,
                1
            )

            used_endpoints.add(i)
            used_endpoints.add(best_j)

    return bridged


# ==========================
# 血管抽出メイン処理
# ==========================

def extract_vessels(green, return_debug=False):
    """
    前処理済み緑チャンネル画像から血管を抽出する。
    return_debug=True の場合、確認用のGチャンネル強調画像も返す。
    """

    original_shape = green.shape

    green_small, scale = resize_for_processing(green)

    fov_mask = compute_fov_mask(green_small)

    green_enhanced = enhance_contrast(green_small)

    green_smooth = cv2.GaussianBlur(
        green_enhanced,
        (3, 3),
        0
    )

    # --- Black-hat ---
    blackhat_total = np.zeros_like(green_smooth)

    for kernel_size in BLACKHAT_KERNEL_SIZES:
        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (kernel_size, kernel_size)
        )

        blackhat = cv2.morphologyEx(
            green_smooth,
            cv2.MORPH_BLACKHAT,
            kernel
        )

        blackhat_total = cv2.max(
            blackhat_total,
            blackhat
        )

    blackhat_normalized = percentile_normalize(blackhat_total)

    # --- Frangi ---
    frangi_img = frangi_vesselness(green_smooth)
    frangi_normalized = percentile_normalize(frangi_img)

    # --- 重み付き統合 ---
    vessel_score = cv2.addWeighted(
        blackhat_normalized,
        0.55,
        frangi_normalized,
        0.45,
        0
    )

    vessel_score = percentile_normalize(vessel_score)

    vessel_score = cv2.bitwise_and(
        vessel_score,
        vessel_score,
        mask=fov_mask
    )

    frangi_normalized_masked = cv2.bitwise_and(
        frangi_normalized,
        frangi_normalized,
        mask=fov_mask
    )

    # --- Hysteresis ---
    vessel_raw = hysteresis_threshold_with_seed(
        score_image=vessel_score,
        seed_image=frangi_normalized_masked
    )

    # --- 接続 ---
    vessel_connected = connect_vessel_segments(vessel_raw)

    vessel_connected = cv2.bitwise_and(
        vessel_connected,
        vessel_connected,
        mask=fov_mask
    )

    # --- 小領域除去 ---
    vessel_small_removed = remove_tiny_components(
        vessel_connected,
        min_area=MIN_COMPONENT_AREA_AFTER_CONNECT
    )

    # --- 細長さフィルタ ---
    vessel_filtered = filter_by_elongation(vessel_small_removed)

    # --- 細線化 ---
    vessel_skeleton = skeletonize_binary(vessel_filtered)

    # --- 枝刈り ---
    vessel_skeleton = prune_short_skeleton_branches(
        skeleton_image=vessel_skeleton,
        vessel_score=vessel_score
    )

    vessel_skeleton = remove_tiny_skeleton_components(
        vessel_skeleton,
        min_pixels=MIN_SKELETON_COMPONENT_PIXELS
    )

    # --- ギャップ橋渡し ---
    if ENABLE_BRIDGE_GAPS:
        vessel_skeleton = bridge_skeleton_gaps(vessel_skeleton)

    # --- デバッグ用Gチャンネル強調画像 ---
    green_enhanced_debug = green_enhanced.copy()

    # --- 元解像度へ戻す ---
    if scale != 1.0:
        vessel_filtered = resize_to_shape(
            vessel_filtered,
            original_shape,
            is_binary=True
        )

        vessel_skeleton = resize_to_shape(
            vessel_skeleton,
            original_shape,
            is_binary=True
        )

        vessel_skeleton = skeletonize_binary(vessel_skeleton)

        green_enhanced_debug = resize_to_shape(
            green_enhanced_debug,
            original_shape,
            is_binary=False
        )

    if return_debug:
        debug_images = {
            "green_enhanced": green_enhanced_debug,
            "vessel_score": resize_to_shape(
                vessel_score,
                original_shape,
                is_binary=False
            ) if scale != 1.0 else vessel_score,
        }

        return vessel_filtered, vessel_skeleton, debug_images

    return vessel_filtered, vessel_skeleton


# ==========================
# オーバーレイ画像作成
# ==========================

def create_overlay(original_img, vessel_skeleton):
    """
    元画像に血管スケルトンを赤色で重ねる。
    """

    overlay = original_img.copy()

    vessel_colored = np.zeros_like(original_img)
    vessel_colored[:, :, 2] = vessel_skeleton

    overlay = cv2.addWeighted(
        overlay,
        0.75,
        vessel_colored,
        0.45,
        0
    )

    return overlay


# ==========================
# ランダム確認用画像保存
# ==========================

def save_random_sample_images(
    image_id,
    original_img,
    green_enhanced,
    vessel_skeleton,
    overlay,
):
    """
    ランダム確認用に、元画像、Gチャンネル強調画像、血管画像、オーバーレイ画像を保存する。
    すべて random_sample_output_dir の1つのフォルダ内に保存する。
    """

    os.makedirs(random_sample_output_dir, exist_ok=True)

    original_output_path = os.path.join(
        random_sample_output_dir,
        f"{image_id}_original.png"
    )

    green_enhanced_output_path = os.path.join(
        random_sample_output_dir,
        f"{image_id}_green_enhanced.png"
    )

    vessel_output_path = os.path.join(
        random_sample_output_dir,
        f"{image_id}_vessel_skeleton.png"
    )

    overlay_output_path = os.path.join(
        random_sample_output_dir,
        f"{image_id}_overlay.png"
    )

    cv2.imwrite(original_output_path, original_img)
    cv2.imwrite(green_enhanced_output_path, green_enhanced)
    cv2.imwrite(vessel_output_path, vessel_skeleton)
    cv2.imwrite(overlay_output_path, overlay)


# ==========================
# メイン処理
# ==========================

def main():
    os.makedirs(mask_output_dir, exist_ok=True)
    os.makedirs(skeleton_output_dir, exist_ok=True)
    os.makedirs(overlay_output_dir, exist_ok=True)
    os.makedirs(random_sample_output_dir, exist_ok=True)

    files = sorted(
        [
            f for f in os.listdir(green_image_dir)
            if f.lower().endswith(".png")
        ]
    )

    if len(files) == 0:
        raise FileNotFoundError(
            f"{green_image_dir} にpng画像がありません。"
        )

    random.seed(RANDOM_SAMPLE_SEED)

    random_sample_files = set(
        random.sample(
            files,
            min(RANDOM_SAMPLE_COUNT, len(files))
        )
    )

    print("ランダム確認用に選ばれた画像:")
    for sample_file in sorted(random_sample_files):
        print(f"  {sample_file}")

    for file in tqdm(files):
        green_path = os.path.join(
            green_image_dir,
            file
        )

        green = cv2.imread(
            green_path,
            cv2.IMREAD_GRAYSCALE
        )

        if green is None:
            print(f"画像を読み込めませんでした: {green_path}")
            continue

        original_file = get_original_file_name(file)

        original_path = os.path.join(
            original_image_dir,
            original_file
        )

        original_img = cv2.imread(original_path)

        if original_img is None:
            original_img = cv2.cvtColor(
                green,
                cv2.COLOR_GRAY2BGR
            )

        vessel_mask, vessel_skeleton, debug_images = extract_vessels(
            green,
            return_debug=True
        )

        image_id = get_image_id(file)

        overlay = create_overlay(
            original_img,
            vessel_skeleton
        )

        cv2.imwrite(
            os.path.join(
                mask_output_dir,
                f"{image_id}_vessel_mask.png"
            ),
            vessel_mask
        )

        cv2.imwrite(
            os.path.join(
                skeleton_output_dir,
                f"{image_id}_vessel_skeleton.png"
            ),
            vessel_skeleton
        )

        cv2.imwrite(
            os.path.join(
                overlay_output_dir,
                f"{image_id}_overlay.png"
            ),
            overlay
        )

        if file in random_sample_files:
            save_random_sample_images(
                image_id=image_id,
                original_img=original_img,
                green_enhanced=debug_images["green_enhanced"],
                vessel_skeleton=vessel_skeleton,
                overlay=overlay
            )

    print("血管抽出が完了しました。")
    print(f"マスク画像: {mask_output_dir}")
    print(f"スケルトン画像: {skeleton_output_dir}")
    print(f"オーバーレイ画像: {overlay_output_dir}")
    print(f"ランダム確認用画像: {random_sample_output_dir}")


if __name__ == "__main__":
    main()

ランダム確認用に選ばれた画像:
  218c822a3dd9_green_preprocessed.png
  3b10191dfd25_green_preprocessed.png
  3f0d3629d69e_green_preprocessed.png
  46acc506fa61_green_preprocessed.png
  5db2e3a4594a_green_preprocessed.png
  65cf00be6fb4_green_preprocessed.png
  6e092caa065f_green_preprocessed.png
  b7a1bb106051_green_preprocessed.png
  d516f77d4516_green_preprocessed.png
  de6210f88536_green_preprocessed.png


100%|██████████| 1434/1434 [22:36<00:00,  1.06it/s]

血管抽出が完了しました。
マスク画像: vessel_output/masks
スケルトン画像: vessel_output/skeletons
オーバーレイ画像: vessel_output/overlays
ランダム確認用画像: vessel_output/random_samples
